In [0]:
def transform_provider(df):
    df=df.dropDuplicates(['provider'])
    return df


In [0]:
from pyspark.sql.functions import (
    col,
    when,
    current_timestamp,
    concat_ws
)

def validate_provider(df):

    invalid_condition = (
        col("Provider").isNull()
        |
        col("PotentialFraud").isNull()
        |
        (~col("PotentialFraud").isin("Yes", "No"))
    )

    valid_df = df.filter(~invalid_condition)

    reject_df = (
        df.filter(invalid_condition)
          .withColumn(
              "reject_reason",
              concat_ws(
                  "; ",
                  when(col("Provider").isNull(), "Provider is NULL"),
                  when(col("PotentialFraud").isNull(), "PotentialFraud is NULL"),
                  when(
                      ~col("PotentialFraud").isin("Yes", "No"),
                      "PotentialFraud must be Yes or No"
                  )
              )
          )
          .withColumn(
              "reject_ts",
              current_timestamp()
          )
    )

    return valid_df, reject_df